                          Project: Berlin Lakes 

My  notebook processes multiple data 

 Step 1 — Setup & Imports
Loaded required libraries and defined the /sources directory.
 
 Step 2 — Fetch OSM Lakes (Berlin)
downloaded fresh lake geometries from the Overpass API, saving it as:
osm_berlin_lakes_raw.json → raw Overpass response
osm_berlin_lakes.geojson → converted GeoJSON polygons

 Step 3 — Convert Raw OSM JSON → GeoJSON
Transformed OSM way geometries into proper polygon features.

Step 4 — Inspect the Clean OSM GeoJSON
Basic sanity checks:
number of lakes
missing names
unique natural and water types
This step ensures transparency of the OSM dataset (per Kai).

Step 5 — Cleaned & Harmonized OSM Data
Standardized column names
Removed unnamed lakes
Normalize name strings
Compute centroids
Compute area (ha) after projecting CRS

Export: berlin_lakes_clean.geojson

 Step 6 — Cleaned Dämeritzsee in-situ measurements
Load demeritzsee.csv
Standardize columns
Drop empty rows, remove duplicates
Export: demeritzsee_clean.csv

Step 7 — Build Unified Dataset
Combines OSM + Dämeritzsee:
Add metadata fields (data_source, last_updated)
Match lakes where name contains “dämeritz”
Export:
berlin_lakes_summary.csv → tabular summary
lakes_berlin_unified.geojson → final dataset

 Step 8 — Final Sanity Checks

Ensuresd that;

Required files exist
Shape consistency
Unified dataset contains geometry + metadata

📁 Output Files (All stored in /sources)
File	Description
osm_berlin_lakes_raw.json	Raw Overpass API output
osm_berlin_lakes.geojson	Clean OSM GeoJSON polygons
berlin_lakes_clean.geojson	Harmonized OSM lake dataset
demeritzsee.csv	Raw in-situ CSV
demeritzsee_clean.csv	Cleaned in-situ measurements
berlin_lakes_summary.csv	Summary without geometry
lakes_berlin_unified.geojson	Final unified dataset

In [32]:
#  Importing and folder setup

from pathlib import Path
import os
import json
import requests

import pandas as pd
import geopandas as gpd


SRC = Path("../sources").resolve()

print("Sources folder:", SRC)
print("Existing files in sources/:")
for f in sorted(SRC.glob("*")):
    print("  -", f.name)


Sources folder: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources
Existing files in sources/:
  - .DS_Store
  - README.md
  - berlin_lakes_clean.geojson
  - berlin_lakes_summary.csv
  - demeritzsee.csv
  - demeritzsee_clean.csv
  - lakes_berlin_unified.geojson
  - osm_berlin_lakes.geojson
  - osm_berlin_lakes_raw.json
  - scripts


In [34]:
#  Fetching OSM Berlin lakes (raw JSON)

raw_path = SRC / "osm_berlin_lakes_raw.json"
overpass_url = "https://overpass-api.de/api/interpreter"

# Overpass query, bringing all waterbodies in Berlin that are lakes/ponds
query = """
[out:json][timeout:180];
area[name="Berlin"]["boundary"="administrative"]->.searchArea;

(
  way["natural"="water"]["water"~"lake|pond"](area.searchArea);
  relation["natural"="water"]["water"~"lake|pond"](area.searchArea);
);

out geom;
"""

if raw_path.exists():
    print("Raw OSM file already exists, skipping download:")
    print(" ", raw_path)
else:
    print("Fetching OSM lakes from Overpass…")
    resp = requests.get(overpass_url, params={"data": query})
    resp.raise_for_status()
    data = resp.json()

    with raw_path.open("w", encoding="utf-8") as f:
        json.dump(data, f)

    print("✅ Saved raw Overpass JSON to:", raw_path)


Raw OSM file already exists, skipping download:
  /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json


In [36]:
# I  Convert raw JSON → clean GeoJSON 

raw_path = SRC / "osm_berlin_lakes_raw.json"
geojson_path = SRC / "osm_berlin_lakes.geojson"

print("Reading raw OSM JSON from:", raw_path)

with raw_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

elements = data.get("elements", [])
features = []

# Convert ways to polygons
for el in elements:
    if el.get("type") == "way" and "geometry" in el:
        coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]

        # Ensuring that polygon is closed
        if len(coords) >= 4 and coords[0] != coords[-1]:
            coords.append(coords[0])

        tags = el.get("tags", {})
        features.append(
            {
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [coords],
                },
                "properties": {
                    "name": tags.get("name"),
                    "natural": tags.get("natural"),
                    "water": tags.get("water"),
                    "wikidata": tags.get("wikidata"),
                },
            }
        )

geojson = {
    "type": "FeatureCollection",
    "name": "osm_berlin_lakes",
    "features": features,
}

with geojson_path.open("w", encoding="utf-8") as f:
    json.dump(geojson, f)

print(f"✅ Created clean GeoJSON with {len(features)} features → {geojson_path}")


Reading raw OSM JSON from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json
✅ Created clean GeoJSON with 633 features → /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson


In [37]:
#  Loading & inspecting clean OSM GeoJSON (Berlin lakes)

# Path to the clean GeoJSON  ( created in  Cell 3)
geojson_path = SRC / "osm_berlin_lakes.geojson"

print("Loading clean OSM GeoJSON from:", geojson_path)

if not geojson_path.exists():
    raise FileNotFoundError(f"GeoJSON not found at {geojson_path}. Run Cell 3 first.")

import geopandas as gpd

gdf_osm = gpd.read_file(geojson_path)

print("\nGeoDataFrame loaded.")
print("Shape:", gdf_osm.shape)
print("Columns:", list(gdf_osm.columns))

print("\nFirst 5 rows:")
print(gdf_osm.head())

# doing my  Quick QA checks on OSM lakes ---
print("\nQA checks:")
print("- Number of lakes with missing 'name':", gdf_osm['name'].isna().sum())
print("- Unique 'natural' values:", gdf_osm['natural'].dropna().unique())
print("- Unique 'water' values:", gdf_osm['water'].dropna().unique())


Loading clean OSM GeoJSON from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson

GeoDataFrame loaded.
Shape: (633, 5)
Columns: ['name', 'natural', 'water', 'wikidata', 'geometry']

First 5 rows:
            name natural water  wikidata  \
0  Schlachtensee   water  lake   Q896095   
1     Koenigssee   water  lake  Q1778289   
2        Fennsee   water  lake  Q1404799   
3      Dreipfuhl   water  pond      None   
4     Lietzensee   water  lake   Q258569   

                                            geometry  
0  POLYGON ((13.2281 52.44468, 13.22814 52.44463,...  
1  POLYGON ((13.27181 52.48856, 13.27203 52.48864...  
2  POLYGON ((13.31548 52.48379, 13.31556 52.48379...  
3  POLYGON ((13.27313 52.44797, 13.27316 52.44798...  
4  POLYGON ((13.289 52.50765, 13.28874 52.50756, ...  

QA checks:
- Number of lakes with missing 'name': 379
- Unique 'natural' values: ['water']
- Unique 'water' values: ['lake' 'pond']


In [39]:
#   Cleaning & harmonizing OSM lake data

from pathlib import Path
import geopandas as gpd

clean_geojson = SRC / "berlin_lakes_clean.geojson"

print("Loading GeoJSON:", geojson_path)
gdf = gpd.read_file(geojson_path)

print("Before cleaning:", gdf.shape)

#  Standardize column names ---
gdf = gdf.rename(columns={
    "name": "lake_name",
    "natural": "natural_type",
    "water": "water_type"
})

# -- Removing lakes with no name (Kai complained about this) ---
gdf = gdf.dropna(subset=["lake_name"])
print("After removing unnamed lakes:", gdf.shape)

# --- Normalizing names (avoid strange characters) ---
gdf["lake_name"] = (
    gdf["lake_name"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# ---  Computing centroids ---
gdf["centroid_lon"] = gdf.geometry.centroid.x
gdf["centroid_lat"] = gdf.geometry.centroid.y

# --- Computing area in hectares ---
# First ensure projected CRS (for metric accuracy)
gdf_metric = gdf.to_crs(epsg=25833)
gdf["area_ha"] = gdf_metric.area / 10_000

print("Columns after engineering:", gdf.columns.tolist())

# ---  Saving cleaned GeoJSON ---
gdf.to_file(clean_geojson, driver="GeoJSON")
print("✅ Saved cleaned lakes to:", clean_geojson)


Loading GeoJSON: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson
Before cleaning: (633, 5)
After removing unnamed lakes: (254, 5)
Columns after engineering: ['lake_name', 'natural_type', 'water_type', 'wikidata', 'geometry', 'centroid_lon', 'centroid_lat', 'area_ha']
✅ Saved cleaned lakes to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_45124/1035893653.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid_lon"] = gdf.geometry.centroid.x
/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_45124/1035893653.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid_lat"] = gdf.geometry.centroid.y


In [43]:
#  – Loading & clean Dämeritzsee in-situ data (demeritzsee.csv)

from pathlib import Path
import pandas as pd

csv_path = SRC / "demeritzsee.csv"
clean_csv_path = SRC / "demeritzsee_clean.csv"

print("Loading raw Dämeritzsee CSV from:", csv_path)

df = pd.read_csv(csv_path)

print("Raw Dämeritzsee shape:", df.shape)
print("Raw columns:", df.columns.tolist())

# . Standardizing column names (lowercase, no spaces)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print("Standardized columns:", df.columns.tolist())

# . Droping fully empty rows / columns
df = df.dropna(how="all")
df = df.loc[:, df.notna().any(axis=0)]
print("After removing empty rows/cols:", df.shape)

# . Removing duplicate rows
before = df.shape[0]
df = df.drop_duplicates()
print("Removed duplicates:", before - df.shape[0])
print("Final cleaned shape:", df.shape)

#  Saving cleaned CSV
df.to_csv(clean_csv_path, index=False)
print("✅ Saved cleaned Dämeritzsee data to:", clean_csv_path)


Loading raw Dämeritzsee CSV from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/demeritzsee.csv
Raw Dämeritzsee shape: (203, 1)
Raw columns: ['Download Reference Id: 2260;;;;;;;;']
Standardized columns: ['download_reference_id:_2260;;;;;;;;']
After removing empty rows/cols: (203, 1)
Removed duplicates: 1
Final cleaned shape: (202, 1)
✅ Saved cleaned Dämeritzsee data to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/demeritzsee_clean.csv


In [45]:
# – Building unified lake dataset & export CSV + GeoJSON

from pathlib import Path
from datetime import datetime
import geopandas as gpd
import pandas as pd

SRC = Path("../sources").resolve()

clean_geo_path   = SRC / "berlin_lakes_clean.geojson"
dem_clean_path   = SRC / "demeritzsee_clean.csv"
summary_csv_path = SRC / "berlin_lakes_summary.csv"
unified_geo_path = SRC / "lakes_berlin_unified.geojson"

print("Loading cleaned lakes from:", clean_geo_path)
gdf = gpd.read_file(clean_geo_path)
print("Cleaned lakes shape:", gdf.shape)
print("Columns:", list(gdf.columns))

# ---- Adding metadata columns ----
timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"
gdf["data_source"]  = "OSM waterbodies (Berlin)"
gdf["last_updated"] = timestamp

# If we have cleaned Dämeritzsee data, mark that lake in data_source
if dem_clean_path.exists():
    dem_df = pd.read_csv(dem_clean_path)
    print("Loaded cleaned Dämeritzsee data:", dem_df.shape)

    mask = gdf["lake_name"].str.contains("ämeritz", case=False, na=False)
    num_marked = mask.sum()
    print("Lakes matched as Dämeritzsee:", num_marked)

    if num_marked > 0:
        gdf.loc[mask, "data_source"] = "OSM + demeritzsee in-situ"

# ---- Exporingt summary CSV (without geometry) ----
summary_df = gdf.drop(columns="geometry")
summary_df.to_csv(summary_csv_path, index=False)
print("✅ Saved berlin_lakes_summary.csv to:", summary_csv_path)

# ---- Exporting unified GeoJSON (with geometry + metadata) ----
gdf.to_file(unified_geo_path, driver="GeoJSON")
print("✅ Saved lakes_berlin_unified.geojson to:", unified_geo_path)


Loading cleaned lakes from: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson
Cleaned lakes shape: (254, 8)
Columns: ['lake_name', 'natural_type', 'water_type', 'wikidata', 'centroid_lon', 'centroid_lat', 'area_ha', 'geometry']
Loaded cleaned Dämeritzsee data: (202, 1)
Lakes matched as Dämeritzsee: 0
✅ Saved berlin_lakes_summary.csv to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv
✅ Saved lakes_berlin_unified.geojson to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/lakes_berlin_unified.geojson


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_45124/558089676.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"


In [47]:
#  – Final sanity checks on outputs

from pathlib import Path
import pandas as pd
import geopandas as gpd

SRC = Path("../sources").resolve()

files = {
    "raw_osm": SRC / "osm_berlin_lakes_raw.json",
    "osm_geojson": SRC / "osm_berlin_lakes.geojson",
    "clean_osm_geojson": SRC / "berlin_lakes_clean.geojson",
    "demeritz_raw": SRC / "demeritzsee.csv",
    "demeritz_clean": SRC / "demeritzsee_clean.csv",
    "summary_csv": SRC / "berlin_lakes_summary.csv",
    "unified_geojson": SRC / "lakes_berlin_unified.geojson",
}

print("=== File existence ===")
for name, path in files.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:20s} {status:7s} -> {path}")

print("\n=== Dataset sizes ===")
gdf_clean = gpd.read_file(files["clean_osm_geojson"])
print("Clean OSM lakes:          ", gdf_clean.shape)

dem_clean = pd.read_csv(files["demeritz_clean"])
print("Clean Dämeritzsee rows:   ", dem_clean.shape)

summary_df = pd.read_csv(files["summary_csv"])
print("Summary CSV:              ", summary_df.shape)

gdf_unified = gpd.read_file(files["unified_geojson"])
print("Unified GeoJSON:          ", gdf_unified.shape)


=== File existence ===
raw_osm              OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json
osm_geojson          OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson
clean_osm_geojson    OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson
demeritz_raw         OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/demeritzsee.csv
demeritz_clean       OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/demeritzsee_clean.csv
summary_csv          OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv
unified_geojson      OK      -> /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/lakes_berlin_unified.geojson

=== Dataset sizes ===
Clean OSM lakes:           (254, 8)
Clean 